In [1]:
import os

In [2]:
%pwd

'c:\\Users\\adity\\DL-Project---Chicken-Disease-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\adity\\DL-Project---Chicken-Disease-Classification'

##### Updating the Entity
-  Entity is a return type of the function which is used to return the value of the function
-  It is used to return the value of the function to the caller function

In [5]:
# importing the data ingestion config class so that we can use it to create an object of the class and pass the required parameters to it.

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
# importing the constants file to get the required constants from it.
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories # we will use this function to create the required directories for our project.

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH, # this is the path to the config.yaml file which contains all the configurations for our project.
            params_filepath = PARAMS_FILE_PATH # this is the path to the params.yaml file which contains all the parameters for our project.
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root]) # this creates the required directories for our project.

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion # this is the data_ingestion section of the config.yaml file which contains all the configurations for the data ingestion process.

        create_directories([config.root_dir])

        # this creates an object of the DataIngestionConfig class and passes the required parameters to it.
        data_ingestion_config = DataIngestionConfig (
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

#### Updating the Components



In [8]:
import os
import urllib.request as request
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__ (self, config: DataIngestionConfig):
        self.config = config

    def download_file(self): # this function is responsible for downloading the file from the source URL and saving it to the local data file path.
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(self.config.source_URL, self.config.local_data_file)
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self): # this function is responsible for extracting the zip file to the unzip directory.
        os.makedirs(self.config.unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

#### Creating Pipeline

In [10]:
try:
    config = ConfigurationManager() # this creates an object of the ConfiguationManager class which will be responsible for managing the configurations for our project.
    data_ingestion_config = config.get_data_ingestion_config() # this gets the data ingestion configuration from the config.yaml file and returns an object of the DataIngestionConfig class which contains all the configurations for the data ingestion process.
    data_ingestion = DataIngestion(config=data_ingestion_config) # this creates an object of the DataIngestion class and passes the data ingestion configuration to it.
    data_ingestion.download_file() # this calls the download_file function which will download the file from the source URL and save it to the local data file path.
    data_ingestion.extract_zip_file() # this calls the extract_zip_file function which will extract the zip file to the unzip directory.
except Exception as e: # if there is any exception during the data ingestion process, it will be caught here and logged.
    raise e

[2026-03-22 20:11:23,183: INFO: common: 32 yaml file: config\config.yaml loaded successfully]
[2026-03-22 20:11:23,185: INFO: common: 32 yaml file: params.yaml loaded successfully]
[2026-03-22 20:11:23,186: INFO: common: 56 directory created at: artifacts]
[2026-03-22 20:11:23,187: INFO: common: 56 directory created at: artifacts/data_ingestion]
[2026-03-22 20:11:25,946: INFO: 114641322: 8 artifacts/data_ingestion/data.zip downloaded with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 1818:A8DFD:271CEA3:2C8DB46:69BFFF92
Accept-Ranges: bytes
Date: Sun, 22 Mar 2026 14:41:24 GMT
Via: 1.1 varnish
X-Served-By: cache-bom-van